# Lezione 7D — Soluzioni: ReAct e Context Engineering

**Corso**: Programmazione di Applicazioni Intelligenti  
**Tipo**: Soluzioni dell'esercitazione 7C


In [ ]:
# === Setup ===

!pip install -q openai

from openai import OpenAI
from google.colab import userdata
import json
from datetime import datetime

client = OpenAI(
    base_url="https://api.mistral.ai/v1",
    api_key=userdata.get("MISTRAL_API_KEY"),
)

MODEL = "mistral-small-latest"
print(f"Client configurato — modello: {MODEL}")


In [ ]:
# === Tool e schema (identici all'esercitazione) ===

def search_wikipedia(query: str) -> str:
    knowledge = {
        "leonardo da vinci": "Leonardo di ser Piero da Vinci (1452-1519) e' stato un inventore, artista e scienziato italiano del Rinascimento. Nato a Vinci, in Toscana, il 15 aprile 1452.",
        "albert einstein": "Albert Einstein (1879-1955) e' stato un fisico teorico tedesco. Nato a Ulm il 14 marzo 1879. Premio Nobel per la fisica nel 1921.",
        "python": "Python e' un linguaggio di programmazione ad alto livello creato da Guido van Rossum e rilasciato nel 1991.",
    }
    for key, value in knowledge.items():
        if key in query.lower():
            return value
    return f"Nessun risultato per: {query}"

def get_current_date() -> str:
    now = datetime.now()
    return f"Oggi e' il {now.day}/{now.month}/{now.year}"

def calculate(expression: str) -> str:
    allowed = set("0123456789+-*/.(). ")
    if not all(c in allowed for c in expression):
        return "Errore: espressione non valida"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Errore: {e}"

tools = [
    {"type": "function", "function": {"name": "search_wikipedia", "description": "Cerca informazioni su Wikipedia.", "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Termine di ricerca"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "get_current_date", "description": "Restituisce la data corrente.", "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "calculate", "description": "Calcola un'espressione matematica.", "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "Espressione (es. '2025 - 1452')"}}, "required": ["expression"]}}},
]

tool_registry = {
    "search_wikipedia": search_wikipedia,
    "get_current_date": get_current_date,
    "calculate": calculate,
}


---
## Soluzione Esercizio 1 — Loop ReAct


In [ ]:
# === SOLUZIONE: Loop ReAct ===

def react_agent(question: str, tools: list, tool_registry: dict, max_steps: int = 10) -> str:
    messages = [
        {"role": "system", "content": "Sei un assistente utile. Usa i tool a disposizione per rispondere. Puoi fare piu chiamate in sequenza se necessario."},
        {"role": "user", "content": question}
    ]

    for step in range(1, max_steps + 1):
        print(f"\n--- Step {step} ---")

        # 1. Chiama il modello
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )
        choice = response.choices[0]
        message = choice.message

        # 2. Controlla finish_reason
        if choice.finish_reason == "stop":
            print("[STOP] Risposta finale generata")
            return message.content

        elif choice.finish_reason == "tool_calls":
            print(f"[TOOL CALLS] {len(message.tool_calls)} chiamata/e")
            # Aggiungi il messaggio dell'assistente alla cronologia.
            # Nota: l'SDK accetta direttamente l'oggetto Pydantic; in alternativa
            # si puo' usare message.model_dump() per una forma dict esplicita.
            messages.append(message)

            # 3. Esegui ogni tool call
            for tc in message.tool_calls:
                func_name = tc.function.name
                func_args = json.loads(tc.function.arguments)
                print(f"  -> {func_name}({func_args})")

                if func_name in tool_registry:
                    result = tool_registry[func_name](**func_args)
                else:
                    result = f"Tool '{func_name}' non trovato"

                print(f"  <- {result}")

                # Aggiungi il risultato alla cronologia
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": str(result)
                })
        else:
            print(f"[UNEXPECTED] finish_reason={choice.finish_reason}")
            return message.content or "Errore inatteso"

    return "Timeout: troppi step"


In [ ]:
# === Test Esercizio 1 ===

risposta = react_agent(
    question="In che anno e' nato Leonardo da Vinci e quanti anni fa e' stato?",
    tools=tools,
    tool_registry=tool_registry,
)

print("\n" + "="*60)
print("RISPOSTA FINALE:", risposta)


---
## Soluzione Esercizio 2 — Logging strutturato


### Nota sulla semantica di `total_tokens`

La soluzione somma `response.usage.total_tokens` di ogni chiamata al modello.
Occorre essere consapevoli di cosa rappresenta davvero questa metrica:

- `response.usage.total_tokens` = `prompt_tokens + completion_tokens` **di quella singola chiamata**.
- Ad ogni step ReAct il prompt include tutta la cronologia, quindi `prompt_tokens`
  cresce cumulativamente. Sommando `total_tokens` step per step, stiamo contando
  più volte gli stessi messaggi di input.

**Di conseguenza**: il `total_tokens` che restituiamo è una stima del **costo API
cumulativo** (utile per stimare la spesa, perché ogni chiamata viene fatturata
con il suo prompt completo), *non* il numero di token "unici" nella conversazione.

Per ottenere quest'ultimo basterebbe prendere `response.usage.prompt_tokens`
dell'ultima chiamata più la somma dei `completion_tokens` di ogni step — ma per
i fini di questa esercitazione teniamo la forma più semplice.


In [ ]:
# === SOLUZIONE: Agente con logging ===

def react_agent_with_logging(question: str, tools: list, tool_registry: dict, max_steps: int = 10) -> dict:
    messages = [
        {"role": "system", "content": "Sei un assistente utile. Usa i tool a disposizione."},
        {"role": "user", "content": question}
    ]
    log = []
    total_tokens = 0  # somma dei total_tokens di ogni chiamata (costo API cumulativo)

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        choice = response.choices[0]
        message = choice.message

        step_tokens = response.usage.total_tokens if response.usage else 0
        total_tokens += step_tokens

        step_log = {
            "step": step,
            "finish_reason": choice.finish_reason,
            "tokens_this_step": step_tokens,
            "tool_calls": [],
        }

        if choice.finish_reason == "stop":
            step_log["final_answer"] = message.content
            log.append(step_log)
            return {
                "answer": message.content,
                "steps": len(log),
                "total_tokens": total_tokens,
                "log": log,
            }

        elif choice.finish_reason == "tool_calls":
            messages.append(message)
            for tc in message.tool_calls:
                func_name = tc.function.name
                func_args = json.loads(tc.function.arguments)
                result = tool_registry.get(func_name, lambda **kw: "Tool non trovato")(**func_args)
                step_log["tool_calls"].append({"tool": func_name, "args": func_args, "result": result})
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

        log.append(step_log)

    return {
        "answer": "Timeout: troppi step",
        "steps": len(log),
        "total_tokens": total_tokens,
        "log": log,
    }


In [ ]:
# === Test Esercizio 2 ===

result = react_agent_with_logging(
    question="Cerca informazioni su Albert Einstein e calcola quanti anni aveva quando ha vinto il Nobel.",
    tools=tools, tool_registry=tool_registry
)
print(f"Risposta: {result['answer']}")
print(f"Step: {result['steps']}, Token (costo API cumulativo): {result['total_tokens']}")
for e in result['log']:
    print(f"\nStep {e['step']} ({e['finish_reason']}, {e['tokens_this_step']} tok)")
    for tc in e.get('tool_calls', []):
        print(f"  {tc['tool']}({tc['args']}) -> {tc['result']}")


---
## Soluzione Esercizio 3 — Compressione del contesto

**Scelta implementativa**: inseriamo il riassunto come `role: "user"` con un marker
esplicito `[RIASSUNTO CONVERSAZIONE PRECEDENTE]`. Questo è coerente con l'esempio
mostrato nel notebook 7B ed evita di avere due messaggi `system` consecutivi,
che molti provider accettano ma che è pratica discutibile (il `system` dovrebbe
restare univoco e definire il comportamento dell'agente, non la cronologia).

Alternative accettabili:
- riassunto come `role: "assistant"` (utile se vogliamo "dare la voce" al modello);
- riassunto concatenato nel `system` originale (più compatto, ma rende il system
  dinamico e meno leggibile in debug).


In [ ]:
# === SOLUZIONE: compress_history ===

def count_tokens_approx(messages: list) -> int:
    return sum(len(m.get("content", "") or "") for m in messages) // 3

def compress_history(messages: list, max_tokens: int = 2000, keep_recent: int = 4) -> list:
    if count_tokens_approx(messages) <= max_tokens:
        return messages

    system = messages[0]
    recent = messages[-keep_recent:]
    to_compress = messages[1:-keep_recent]

    if not to_compress:
        return messages

    history_text = "\n".join(f"[{m['role']}]: {m.get('content','')}" for m in to_compress)

    summary_response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Riassumi questa conversazione in modo conciso, mantenendo i fatti importanti. Rispondi SOLO con il riassunto."},
            {"role": "user", "content": history_text}
        ],
        max_tokens=500
    )
    summary = summary_response.choices[0].message.content

    # Inseriamo il riassunto come messaggio user con marker esplicito:
    # - manteniamo un unico system (quello originale);
    # - il marker rende chiaro al modello che si tratta di contesto riassunto.
    return [
        system,
        {"role": "user", "content": f"[RIASSUNTO CONVERSAZIONE PRECEDENTE]\n{summary}"},
    ] + recent


In [ ]:
# === Test Esercizio 3 ===

test_messages = [
    {"role": "system", "content": "Sei un assistente utile e preciso."},
    {"role": "user", "content": "Ciao, sto organizzando un viaggio in Giappone."},
    {"role": "assistant", "content": "Che bello! Quando vorresti partire e per quanto tempo?"},
    {"role": "user", "content": "Marzo, 2 settimane, budget 3000 euro."},
    {"role": "assistant", "content": "Marzo e' perfetto per i ciliegi! Ti consiglio Tokyo, Kyoto, Osaka e Hiroshima. Japan Rail Pass 14 giorni ~310 euro."},
    {"role": "user", "content": "Posso vedere il Monte Fuji a marzo?"},
    {"role": "assistant", "content": "Il Fuji e' coperto di neve a marzo. Puoi ammirarlo da Hakone."},
    {"role": "user", "content": "Ok Hakone. Ostelli o capsule hotel?"},
    {"role": "assistant", "content": "Ostelli 20-30 euro/notte, capsule 30-40. Consiglio un mix."},
    {"role": "user", "content": "Perfetto, facciamo un mix."},
    {"role": "assistant", "content": "Ottimo! Itinerario: Tokyo 4gg, Hakone 1g, Kyoto 3gg, Osaka 2gg, Hiroshima 2gg."},
    {"role": "user", "content": "Ricapitolami tutto con i costi."},
]

print(f"Prima: {len(test_messages)} msg, ~{count_tokens_approx(test_messages)} tok")
compressed = compress_history(test_messages, max_tokens=250, keep_recent=4)
print(f"Dopo:  {len(compressed)} msg, ~{count_tokens_approx(compressed)} tok")
for i, m in enumerate(compressed):
    print(f"  [{i}] {m['role']}: {(m.get('content','') or '')[:100]}...")


---
## Soluzione Esercizio 4 (bonus) — Personalità diverse


In [ ]:
# === SOLUZIONE: ask_with_personality ===

def ask_with_personality(personality_prompt: str, question: str) -> str:
    """
    Gestisce un singolo round di tool call, poi forza una risposta testuale.
    """
    messages = [
        {"role": "system", "content": personality_prompt},
        {"role": "user", "content": question}
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    msg = response.choices[0].message

    # Se il modello vuole chiamare un tool, eseguilo (un solo round, come da spec)
    if response.choices[0].finish_reason == "tool_calls":
        messages.append(msg)
        for tc in msg.tool_calls:
            func_name = tc.function.name
            func_args = json.loads(tc.function.arguments)
            result = tool_registry.get(func_name, lambda **kw: "Tool non trovato")(**func_args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

        # Secondo round SENZA tools: forziamo una risposta testuale finale.
        # Se passassimo di nuovo tools=tools il modello potrebbe concatenare
        # altre chiamate, violando la spec "un solo round".
        response = client.chat.completions.create(model=MODEL, messages=messages)
        msg = response.choices[0].message

    return msg.content

question = "In che anno e' nato Leonardo da Vinci?"

print("=== PROFESSORE ===")
print(ask_with_personality("Sei un professore. Spiega tutto passo per passo con tono didattico.", question))
print("\n=== INGEGNERE ===")
print(ask_with_personality("Sei un ingegnere pragmatico. Rispondi in modo ultra-conciso, solo fatti.", question))
